# Two-Dimensional Ising Model

This tutorial demonstrates how to use `jaxfss` to perform **Neural Scaling Analysis (NSA)** on the two-dimensional ferromagnetic Ising model on a square lattice.


## 1. Theoretical Background

The Binder ratio $U(T, L)$ is a dimensionless quantity defined by the fourth- and second-order magnetization moments:

$$
U(T, L) = 1 - \frac{\langle M^4 \rangle}{3 \langle M^2 \rangle^2}
$$

Because $U(T, L)$ is dimensionless, its critical exponent $c_2$ is identically zero ($c_2 = 0$). The finite-size scaling hypothesis therefore simplifies to:

$$
U(T, L) = F\left[ (T - T_{\mathrm{c}}) L^{c_1} \right]
$$

where:
* $T_{\mathrm{c}}$ is the critical temperature (exact value for the square lattice: $1/T_{\mathrm{c}} = \frac{1}{2}\ln(1+\sqrt{2}) \approx 0.4406868$).
* $c_1 = 1/\nu = 1.0$ is the correlation length exponent.
* $F[\cdot]$ is the unknown universal scaling function.

## 2. Load and Inspect Data

We use `CriticalData` to load the dataset (`docs/data/ising.txt`).

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
from softclip import SoftClip

import jaxfss

# Load Monte Carlo simulation data
dataset = jaxfss.CriticalData.from_file("data/ising.txt")
print(f"Loaded {dataset.n_data} data points.")
print(f"System sizes: {jnp.unique(dataset.Ls)}")

Let's visualize the raw data before scaling collapse.

In [ ]:
plt.figure(figsize=(7, 4.5))
unique_Ls = sorted(list(set(dataset.Ls.flatten().tolist())))
for L in unique_Ls:
    idx = dataset.Ls.flatten() == L
    plt.errorbar(
        dataset.Ts.flatten()[idx],
        dataset.As.flatten()[idx],
        yerr=dataset.As_err.flatten()[idx],
        fmt="o-",
        label=f"$L = {int(L)}$",
        capsize=2
    )

plt.xlabel(r"Inverse Temperature $1/T$", fontsize=12)
plt.ylabel(r"Binder Ratio $U(T, L)$", fontsize=12)
plt.title("Raw Binder Ratio Data", fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Set Up Neural Scaling Analysis

We parameterize the scaling function using `RationalMLP` and define bijectors to enforce physical bounds:
* $c_1 > 0$ via `SoftClip(low=0.0)`
* Scaled $T_{\mathrm{c}} \in [-1, 1]$ via `SoftClip(low=-1.0, high=1.0)`

In [ ]:
mlp = jaxfss.RationalMLP(features=[20, 20, 1])
key = jax.random.PRNGKey(42)
mlp_params = mlp.init(key, jnp.ones((1, 1)))

bij_c1 = SoftClip(low=0.0)
bij_Tc = SoftClip(low=-1.0, high=1.0)

init_params = {
    "mlp": mlp_params,
    "fss": jnp.zeros(2)  # [p_c1, p_Tc]
}

def get_fss_params(params):
    p1, pc = params["fss"]
    c1 = bij_c1.forward(p1)
    scaled_Tc = bij_Tc.forward(pc)
    return c1, scaled_Tc

train_data = dataset.train_data
Ls = train_data["system_size"]
Ts = train_data["temperature"]
As = train_data["observable"]
vAs = train_data["observable_var"]

def loss_fn(params):
    c1, scaled_Tc = get_fss_params(params)
    X = (Ts - scaled_Tc) * (Ls ** c1)
    Y_pred = mlp.apply(params["mlp"], X)
    return jaxfss.NLLLoss(As, Y_pred, vAs)

## 4. Training with Multi-Optimizer

We use separate learning rates for the MLP weights (`1e-3`) and the critical parameters (`1e-2`).

In [ ]:
optimizer = {
    "mlp": optax.adam(learning_rate=1e-3),
    "fss": optax.adam(learning_rate=1e-2)
}
steps = 8000

params, losses, critical_vals = jaxfss.fit(loss_fn, optimizer, init_params, steps)

c1_est, scaled_Tc_est = get_fss_params(params)
Tc_est = float(dataset.bij_temperature.inverse(scaled_Tc_est))
c1_est = float(c1_est)

print("=== Estimation Results ===")
print(f"c_1 (exact = 1.0)       : {c1_est:.5f}")
print(f"1/Tc (exact = 0.44069)  : {Tc_est:.5f}")

## 5. Scaling Collapse and NN Fitting Curve

Now we plot the collapsed data along with the learned neural network scaling function.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Loss curve
ax1.plot(losses)
ax1.set_yscale("log")
ax1.set_xlabel("Step")
ax1.set_ylabel("NLL Loss")
ax1.set_title("Training Loss Curve")
ax1.grid(True, alpha=0.3)

# Scaling collapse
for L in unique_Ls:
    idx = dataset.Ls.flatten() == L
    t = Ts.flatten()[idx]
    s = Ls.flatten()[idx]
    x = (t - scaled_Tc_est) * (s ** c1_est)
    y = As.flatten()[idx]
    yerr = jnp.sqrt(vAs.flatten()[idx])
    ax2.errorbar(x, y, yerr=yerr, fmt="o", label=f"$L={int(L)}$", alpha=0.7, capsize=2)

# Plot NN prediction
x_grid = jnp.linspace(-1.0, 1.0, 200).reshape(-1, 1)
y_pred = mlp.apply(params["mlp"], x_grid)
ax2.plot(x_grid, y_pred, color="black", lw=2, linestyle="--", label="NN $F(X)$")

ax2.set_xlabel(r"Scaled temperature $X = (T - T_{\mathrm{c}}) L^{c_1}$", fontsize=11)
ax2.set_ylabel(r"$Y = U(T, L)$", fontsize=11)
ax2.set_title(f"Data Collapse ($c_1={c1_est:.4f}, 1/T_c={Tc_est:.4f}$)", fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()